In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, Input, regularizers
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split

# Constants
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50
L2_REG = 1e-4

EYE_CLASSES = ['angry', 'fear', 'happy', 'sad']
HAND_CLASSES = ['neutral', 'stress', 'boring']

DATA_DIR_EYE = r"D:\eye_dataset\150_eye_RGB_NEW4classes"
DATA_DIR_HAND = r"D:\HandMomentDataset"


def load_images_from_folder(data_dir, class_names):
    images, labels = [], []
    for idx, cls_name in enumerate(class_names):
        class_dir = os.path.join(data_dir, cls_name)
        files = [f for f in os.listdir(class_dir) if os.path.isfile(os.path.join(class_dir, f))]
        print(f"Loading {len(files)} from {class_dir}")
        for file in files:
            img_path = os.path.join(class_dir, file)
            img = cv2.imread(img_path)
            if img is None:
                continue
            img = cv2.resize(img, IMG_SIZE)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = preprocess_input(img)
            images.append(img)
            labels.append(idx)
    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.int32)


# Load datasets
X_eye, y_eye = load_images_from_folder(DATA_DIR_EYE, EYE_CLASSES)
X_hand, y_hand = load_images_from_folder(DATA_DIR_HAND, HAND_CLASSES)

# Train-test split
X_eye_train, X_eye_test, y_eye_train, y_eye_test = train_test_split(
    X_eye, y_eye, test_size=0.2, stratify=y_eye, random_state=42)

X_hand_train, X_hand_test, y_hand_train, y_hand_test = train_test_split(
    X_hand, y_hand, test_size=0.2, stratify=y_hand, random_state=42)


def create_shared_backbone(input_shape=(*IMG_SIZE, 3)):
    base = MobileNetV3Small(include_top=False, input_shape=input_shape, pooling='avg', weights='imagenet')
    base.trainable = False
    return base


# Define inputs
input_eye = Input(shape=(*IMG_SIZE, 3), name="eye_input")
input_hand = Input(shape=(*IMG_SIZE, 3), name="hand_input")

shared_backbone = create_shared_backbone()

feat_eye = shared_backbone(input_eye)
feat_hand = shared_backbone(input_hand)

# Combine features
combined_features = layers.Concatenate()([feat_eye, feat_hand])
combined_features = layers.BatchNormalization()(combined_features)
combined_features = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(L2_REG))(combined_features)
combined_features = layers.Dropout(0.5)(combined_features)

# Separate classifiers
eye_output = layers.Dense(len(EYE_CLASSES), activation='softmax', name='eye_emotion')(combined_features)
hand_output = layers.Dense(len(HAND_CLASSES), activation='softmax', name='hand_emotion')(combined_features)

# Compile model
model = Model(inputs=[input_eye, input_hand], outputs=[eye_output, hand_output])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss={
        'eye_emotion': 'sparse_categorical_crossentropy',
        'hand_emotion': 'sparse_categorical_crossentropy'
    },
    metrics={
        'eye_emotion': ['accuracy'],
        'hand_emotion': ['accuracy']
    }
)
model.summary()

# Data augmentations
eye_datagen = ImageDataGenerator(
    rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
    shear_range=0.1, zoom_range=0.1, horizontal_flip=True, fill_mode='nearest'
)

hand_datagen = ImageDataGenerator(
    rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
    shear_range=0.1, zoom_range=0.1, horizontal_flip=True, fill_mode='nearest'
)


def paired_generator(datagen_eye, X_eye, datagen_hand, X_hand, y_eye, y_hand, batch_size):
    gen_eye = datagen_eye.flow(X_eye, y_eye, batch_size=batch_size, seed=42)
    gen_hand = datagen_hand.flow(X_hand, y_hand, batch_size=batch_size, seed=42)
    while True:
        batch_eye = next(gen_eye)
        batch_hand = next(gen_hand)
         # Only yield batches when both are full size and batch sizes match
        if batch_eye[0].shape[0] == batch_size and batch_hand[0].shape[0] == batch_size:
            yield ((batch_eye[0], batch_hand[0]), {'eye_emotion': batch_eye[1], 'hand_emotion': batch_hand[1]})
        else:
            # Skip this batch to avoid size mismatch
            continue
train_gen = paired_generator(eye_datagen, X_eye_train, hand_datagen, X_hand_train, y_eye_train, y_hand_train, BATCH_SIZE)
val_gen = paired_generator(eye_datagen, X_eye_test, hand_datagen, X_hand_test, y_eye_test, y_hand_test, BATCH_SIZE)

train_steps = min(len(X_eye_train), len(X_hand_train)) // BATCH_SIZE
val_steps = min(len(X_eye_test), len(X_hand_test)) // BATCH_SIZE

# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

# Train
history = model.fit(
    train_gen,
    validation_data=val_gen,
    steps_per_epoch=train_steps,
    validation_steps=val_steps,
    epochs=EPOCHS,
    callbacks=[early_stopping, reduce_lr]
)


# Prediction utility
def predict_emotions(eye_img_path, hand_img_path, model):
    def preprocess(img_path):
        img = cv2.imread(img_path)
        if img is None:
            return None
        img = cv2.resize(img, IMG_SIZE)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return preprocess_input(img).astype(np.float32)

    eye_img = preprocess(eye_img_path)
    hand_img = preprocess(hand_img_path)
    if eye_img is None or hand_img is None:
        return None

    eye_input = np.expand_dims(eye_img, axis=0)
    hand_input = np.expand_dims(hand_img, axis=0)

    preds = model.predict([eye_input, hand_input])
    eye_pred, hand_pred = preds

    eye_class = EYE_CLASSES[np.argmax(eye_pred[0])]
    hand_class = HAND_CLASSES[np.argmax(hand_pred[0])]

    return eye_class, h==5634dfggfdasddffsdfjnjndssdsfdvcxvckmk# eye_emotion, hand_emotion = predict_emotions('eye1.jpg', 'hand1.jpg', model)
# print(f"Eye Emotion: {eye_emotion}, Hand Emotion: {hand_emotion}")


Loading 150 from D:\eye_dataset\150_eye_RGB_NEW4classes\angry
Loading 150 from D:\eye_dataset\150_eye_RGB_NEW4classes\fear
Loading 150 from D:\eye_dataset\150_eye_RGB_NEW4classes\happy
Loading 150 from D:\eye_dataset\150_eye_RGB_NEW4classes\sad
Loading 150 from D:\HandMomentDataset\neutral
Loading 150 from D:\HandMomentDataset\stress
Loading 150 from D:\HandMomentDataset\boring


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ eye_input           │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hand_input          │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ MobileNetV3Small    │ (None, 576)       │    939,120 │ eye_input[0][0],  │
│ (Functional)        │                   │            │ hand_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1152)      │          0 │ MobileNetV3Small… │
│ (Concatenate)       │                   │            │ MobileNetV3Small… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 1152)      │      4,608 │ concatenate[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    295,168 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ eye_emotion (Dense) │ (None, 4)         │      1,028 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hand_emotion        │ (None, 3)         │        771 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,240,695 (4.73 MB)

 Trainable params: 299,271 (1.14 MB)

 Non-trainable params: 941,424 (3.59 MB)

Epoch 1/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 29s 2s/step - eye_emotion_accuracy: 0.2460 - eye_emotion_loss: 2.1882 - hand_emotion_accuracy: 0.3951 - hand_emotion_loss: 1.6634 - loss: 3.8935 - val_eye_emotion_accuracy: 0.2812 - val_eye_emotion_loss: 1.5278 - val_hand_emotion_accuracy: 0.4062 - val_hand_emotion_loss: 1.1952 - val_loss: 2.7650 - learning_rate: 1.0000e-04
Epoch 2/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - eye_emotion_accuracy: 0.1910 - eye_emotion_loss: 2.4597 - hand_emotion_accuracy: 0.4162 - hand_emotion_loss: 1.5005 - loss: 4.0022 - val_eye_emotion_accuracy: 0.2969 - val_eye_emotion_loss: 1.4658 - val_hand_emotion_accuracy: 0.4531 - val_hand_emotion_loss: 1.1206 - val_loss: 2.6284 - learning_rate: 1.0000e-04
Epoch 3/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 15s 1s/step - eye_emotion_accuracy: 0.3331 - eye_emotion_loss: 2.1860 - hand_emotion_accuracy: 0.4138 - hand_emotion_loss: 1.5117 - loss: 3.7397 - val_eye_emotion_accuracy: 0.2969 - val_eye_emotion_loss: 1.4268 - val_hand_emotion_accu

In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, Input, regularizers
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50
L2_REG = 1e-4

EYE_CLASSES = ['angry', 'happy', 'fear', 'sad']
HAND_CLASSES = ['Boring', 'Neutral', 'Stress']

DATA_DIR_EYE = r"D:\eye_dataset\150_eye_RGB_NEW4classes"
DATA_DIR_HAND = r"D:\HandMomentDataset"

yolo_eye = YOLO("yolov8n.pt")
yolo_hand = YOLO("yolov8n.pt")

def detect_and_crop(image_path, yolo_model, target_size):
    img = cv2.imread(image_path)
    if img is None:
        return None
    results = yolo_model(img)
    boxes = [box for r in results for box in r.boxes.xyxy.cpu().numpy()]
    if not boxes:
        return cv2.resize(img, target_size)
    largest_box = max(boxes, key=lambda b: (b[2] - b[0]) * (b[3] - b[1]))
    x1, y1, x2, y2 = map(int, largest_box)
    margin_x, margin_y = int((x2-x1)*0.2), int((y2-y1)*0.2)
    x1, y1 = max(0, x1-margin_x), max(0, y1-margin_y)
    x2, y2 = min(img.shape[1], x2+margin_x), min(img.shape[0], y2+margin_y)
    crop = img[y1:y2, x1:x2]
    return cv2.resize(crop, target_size)

def load_data(data_dir, class_names, yolo_model):
    images, labels = [], []
    for idx, cls in enumerate(class_names):
        cls_path = os.path.join(data_dir, cls)
        files = os.listdir(cls_path)
        print(f"Loading {len(files)} images from class '{cls}'")
        for file in files:
            img_path = os.path.join(cls_path, file)
            crop = detect_and_crop(img_path, yolo_model, IMG_SIZE)
            if crop is not None:
                rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                preprocessed = preprocess_input(rgb)
                images.append(preprocessed)
                labels.append(idx)
    images = np.array(images, dtype=np.float32)
    labels = np.array(labels, dtype=np.int32)
    print(f"Loaded {len(images)} images from {data_dir}")
    return images, labels

print("Loading eye train dataset...")
X_eye, y_eye = load_data(DATA_DIR_EYE, EYE_CLASSES, yolo_eye)
X_eye_train, X_eye_test, y_eye_train, y_eye_test = train_test_split(X_eye, y_eye, test_size=0.2, stratify=y_eye, random_state=42)

print("Loading hand dataset...")
X_hand, y_hand = load_data(DATA_DIR_HAND, HAND_CLASSES, yolo_hand)
X_hand_train, X_hand_test, y_hand_train, y_hand_test = train_test_split(X_hand, y_hand, test_size=0.2, stratify=y_hand, random_state=42)

input_eye = Input(shape=(*IMG_SIZE, 3), name="eye_input")
input_hand = Input(shape=(*IMG_SIZE, 3), name="hand_input")

# Instantiate base models separately
base_eye_model = MobileNetV2(include_top=False, input_shape=(*IMG_SIZE, 3), pooling='avg', weights='imagenet', name="mobilenetv2_eye")
base_hand_model = MobileNetV2(include_top=False, input_shape=(*IMG_SIZE, 3), pooling='avg', weights='imagenet', name="mobilenetv2_hand")

# Unfreeze last 30 layers for fine-tuning
for layer in base_eye_model.layers[-30:]:
    layer.trainable = True
for layer in base_hand_model.layers[-30:]:
    layer.trainable = True

# Call models on inputs
base_eye = base_eye_model(input_eye)
base_hand = base_hand_model(input_hand)

eye_feat = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(base_eye)
eye_feat = layers.Dropout(0.5)(eye_feat)
eye_feat = layers.BatchNormalization()(eye_feat)
eye_output = layers.Dense(len(EYE_CLASSES), activation='softmax', name='eye_emotion')(eye_feat)

hand_feat = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(base_hand)
hand_feat = layers.Dropout(0.5)(hand_feat)
hand_feat = layers.BatchNormalization()(hand_feat)
hand_output = layers.Dense(len(HAND_CLASSES), activation='softmax', name='hand_emotion')(hand_feat)

model = Model(inputs=[input_eye, input_hand], outputs=[eye_output, hand_output])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss={
        'eye_emotion': 'sparse_categorical_crossentropy',
        'hand_emotion': 'sparse_categorical_crossentropy'
    },
    metrics={
        'eye_emotion': ['accuracy'],
        'hand_emotion': ['accuracy']
    }
)

model.summary()

train_datagen = ImageDataGenerator(
    rotation_range=20, shear_range=0.1, zoom_range=0.1,
    width_shift_range=0.1, height_shift_range=0.1,
    horizontal_flip=True, fill_mode='nearest'
)
val_datagen = ImageDataGenerator()

def paired_generator(datagen1, x1, datagen2, x2, y1, y2, batch_size):
    gen1 = datagen1.flow(x1, y1, batch_size=batch_size, seed=42)
    gen2 = datagen2.flow(x2, y2, batch_size=batch_size, seed=42)
    while True:
        X1i = next(gen1)
        X2i = next(gen2)
        yield (X1i[0], X2i[0]), {'eye_emotion': X1i[1], 'hand_emotion': X2i[1]}

train_gen = paired_generator(train_datagen, X_eye_train, train_datagen, X_hand_train, y_eye_train, y_hand_train, BATCH_SIZE)

output_signature = (
    (
        tf.TensorSpec(shape=(None, *IMG_SIZE, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None, *IMG_SIZE, 3), dtype=tf.float32),
    ),
    {
        'eye_emotion': tf.TensorSpec(shape=(None,), dtype=tf.int32),
        'hand_emotion': tf.TensorSpec(shape=(None,), dtype=tf.int32),
    }
)

val_dataset = tf.data.Dataset.from_generator(
    lambda: paired_generator(val_datagen, X_eye_test, val_datagen, X_hand_test, y_eye_test, y_hand_test, BATCH_SIZE),
    output_signature=output_signature
)

early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, verbose=1)

history = model.fit(
    train_gen,
    validation_data=val_dataset,
    steps_per_epoch=min(len(X_eye_train), len(X_hand_train)) // BATCH_SIZE,
    validation_steps=min(len(X_eye_test), len(X_hand_test)) // BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=[early_stopping, reduce_lr]
)


Loading eye train dataset...
Loading 150 images from class 'angry'

0: 320x640 (no detections), 1259.4ms
Speed: 81.4ms preprocess, 1259.4ms inference, 52.6ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 (no detections), 61.8ms
Speed: 2.5ms preprocess, 61.8ms inference, 0.7ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 (no detections), 64.8ms
Speed: 2.4ms preprocess, 64.8ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 1 person, 92.3ms
Speed: 2.1ms preprocess, 92.3ms inference, 70.3ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 (no detections), 121.4ms
Speed: 2.1ms preprocess, 121.4ms inference, 1.9ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 (no detections), 93.5ms
Speed: 3.7ms preprocess, 93.5ms inference, 0.9ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 (no detections), 107.2ms
Speed: 2.1ms preprocess, 107.2ms inference, 1.1ms postprocess per image at shape (1, 3, 320, 640

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ eye_input           │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hand_input          │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_eye     │ (None, 1280)      │  2,257,984 │ eye_input[0][0]   │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_hand    │ (None, 1280)      │  2,257,984 │ hand_input[0][0]  │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │    163,968 │ mobilenetv2_eye[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │    163,968 │ mobilenetv2_hand… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128)       │        512 │ dropout[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dropout_1[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ eye_emotion (Dense) │ (None, 4)         │        516 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hand_emotion        │ (None, 3)         │        387 │ batch_normalizat… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,845,831 (18.49 MB)

 Trainable params: 4,777,095 (18.22 MB)

 Non-trainable params: 68,736 (268.50 KB)

Epoch 1/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 140s 8s/step - eye_emotion_accuracy: 0.2481 - eye_emotion_loss: 1.8632 - hand_emotion_accuracy: 0.3554 - hand_emotion_loss: 1.5847 - loss: 3.4945 - val_eye_emotion_accuracy: 0.2656 - val_eye_emotion_loss: 1.5619 - val_hand_emotion_accuracy: 0.4062 - val_hand_emotion_loss: 1.0728 - val_loss: 2.6812 - learning_rate: 1.0000e-04
Epoch 2/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 83s 8s/step - eye_emotion_accuracy: 0.2492 - eye_emotion_loss: 1.7831 - hand_emotion_accuracy: 0.4958 - hand_emotion_loss: 1.2362 - loss: 3.0659 - val_eye_emotion_accuracy: 0.2500 - val_eye_emotion_loss: 1.4811 - val_hand_emotion_accuracy: 0.5781 - val_hand_emotion_loss: 0.9157 - val_loss: 2.4433 - learning_rate: 1.0000e-04
Epoch 3/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 85s 8s/step - eye_emotion_accuracy: 0.3657 - eye_emotion_loss: 1.5132 - hand_emotion_accuracy: 0.6819 - hand_emotion_loss: 0.7086 - loss: 2.2684 - val_eye_emotion_accuracy: 0.3125 - val_eye_emotion_loss: 1.4743 - val_hand_emotion_acc